# LayoutLMv3 Modified (`pn-minh/layoutlmv3-1`) — A100 High-RAM Training & Ablation Suite

**Execution Order (3 Seeds `42, 123, 1993` per run):**
1. **Setup:** A100 GPU & High-RAM verification $\to$ Pull `https://github.com/pn-minh/layoutlmv3-1` $\to$ Miniconda (`python=3.7.16`) virtual environment $\to$ Download & verify `FUNSD` & `CORD`.
2. **Phase A (`layoutlmv3-base`):** Baseline (`FUNSD`, `CORD`) $\to$ Full Model (`FUNSD`, `CORD`) $\to$ Without Boundary Loss (`FUNSD`, `CORD`) $\to$ Without Token-Segment (`FUNSD`, `CORD`) $\to$ Without HPE (`FUNSD`, `CORD`).
3. **Phase B (`layoutlmv3-large`):** Baseline (`FUNSD`, `CORD`) $\to$ Full Model (`FUNSD`, `CORD`) $\to$ Without Boundary Loss (`FUNSD`, `CORD`) $\to$ Without Token-Segment (`FUNSD`, `CORD`) $\to$ Without HPE (`FUNSD`, `CORD`).
4. **Summary:** Aggregate all 3-seed `Mean ± Std` metrics into a final comparison table.

In [ ]:
# Cell 1: Verify A100 GPU + High-RAM Runtime and Clone/Pull Repository from GitHub
# Estimated time to finish this cell: ~10-15 seconds
import os, subprocess, time, psutil

cell_start = time.time()
print("[CELL 1] Starting hardware check & repository sync (Estimated time: ~10-15s)...", flush=True)

# Check GPU model and total system RAM
subprocess.run(["nvidia-smi"], check=False)
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f"[HARDWARE] Total System RAM: {ram_gb:.2f} GB", flush=True)

# Clone or update https://github.com/pn-minh/layoutlmv3-1 in /content/layoutlmv3-1
REPO_DIR = "/content/layoutlmv3-1"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/pn-minh/layoutlmv3-1.git", REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], check=True)

print(f"[CELL 1 COMPLETE] Finished in {time.time() - cell_start:.1f}s", flush=True)

In [ ]:
# Cell 2: Install Miniconda3, Create Python 3.7 Virtual Environment, and Install Compatible Packages
# Estimated time to finish this cell: ~3-5 minutes
import subprocess, sys, time

def run_streaming(cmd: str, est_desc: str):
    """Stream shell command stdout/stderr line-by-line in real time with elapsed time tracking."""
    t0 = time.time()
    print(f"[START] {est_desc}", flush=True)
    proc = subprocess.Popen(
        ["bash", "-c", cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        universal_newlines=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    ret = proc.wait()
    elapsed = time.time() - t0
    if ret != 0:
        raise RuntimeError(f"Command exited with code {ret} after {elapsed:.1f}s")
    print(f"[DONE] Completed in {elapsed:.1f}s ({elapsed/60.0:.2f} min)", flush=True)

setup_script = r'''
set -e
# 1. Download and install Miniconda3 if not already installed
if [ ! -d "/content/miniconda3" ]; then
    wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /content/miniconda.sh
    bash /content/miniconda.sh -b -p /content/miniconda3
    rm -f /content/miniconda.sh
fi

source /content/miniconda3/etc/profile.d/conda.sh
conda config --set always_yes yes
# Accept conda TOS if required by newer Miniconda versions
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main 2>/dev/null || true
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r 2>/dev/null || true

# 2. Create layoutlmv3 conda environment with Python 3.7.16
if ! conda info --envs | grep -q "^layoutlmv3 "; then
    conda create -n layoutlmv3 python=3.7.16 -y
fi
conda activate layoutlmv3

# 3. Upgrade pip/setuptools/wheel to last Python 3.7 compatible versions
python -m pip install --upgrade "pip<24.1" "setuptools<68.0" "wheel<0.43"

# 4. Install PyTorch 1.10.0+cu111 and TorchVision 0.11.1+cu111 (supports A100 sm_80)
python -m pip install torch==1.10.0+cu111 torchvision==0.11.1+cu111 -f https://download.pytorch.org/whl/cu111/torch_stable.html

# 5. Install pinned compatible requirements and editable layoutlmv3 package
cd /content/layoutlmv3-1/layoutlmv3
python -m pip install -r requirements.txt
python -m pip install -e .

# 6. Verify Python, PyTorch, CUDA A100 device, and key libraries
python - << 'PY'
import sys, torch, transformers, datasets, timm, seqeval, PIL
print("=" * 60)
print("Python version      :", sys.version.split()[0])
print("PyTorch version     :", torch.__version__)
print("CUDA available      :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device          :", torch.cuda.get_device_name(0))
print("Transformers version:", transformers.__version__)
print("Datasets version    :", datasets.__version__)
print("timm version        :", timm.__version__)
print("Pillow version      :", PIL.__version__)
print("=" * 60)
PY
'''

run_streaming(setup_script, "Cell 2: Miniconda Python 3.7 + PyTorch 1.10.0+cu111 + Requirements Setup (Est. ~3-5 min)")

In [ ]:
# Cell 3: Download and Verify FUNSD & CORD Datasets + Pre-cache Pretrained Backbones
# Estimated time to finish this cell: ~2-3 minutes
prep_cmd = r'''
set -e
source /content/miniconda3/etc/profile.d/conda.sh
conda activate layoutlmv3
export PYTHONUNBUFFERED=1
cd /content/layoutlmv3-1/layoutlmv3
python scripts/prepare_datasets.py
'''
run_streaming(prep_cmd, "Cell 3: Prepare FUNSD & CORD Datasets (Est. ~2-3 min)")

---
## Phase A: All `layoutlmv3-base` Runs (3 Seeds: `42, 123, 1993`)
Each cell below executes 3 seeds (1,000 steps per seed) on the A100 GPU, streaming real-time progress and ETA every 20 steps, and saves the 3-seed `Mean ± Std` summary JSON.

In [ ]:
# Cell 4: [BASE 1/10] FUNSD — Baseline (Vanilla LayoutLMv3-base, 3 seeds)
# Estimated time on A100: ~6-8 minutes (3 seeds x ~2-2.5 min/seed)
run_streaming(
    "source /content/miniconda3/etc/profile.d/conda.sh && conda activate layoutlmv3 && export PYTHONUNBUFFERED=1 && cd /content/layoutlmv3-1/layoutlmv3 && bash scripts/run_ablation_suite.sh funsd base baseline",
    "Cell 4: FUNSD | BASE | BASELINE (3 seeds: 42, 123, 1993) — Est. ~6-8 min"
)

In [ ]:
# Cell 5: [BASE 2/10] CORD — Baseline (Vanilla LayoutLMv3-base, 3 seeds)
# Estimated time on A100: ~18-22 minutes (3 seeds x ~6-7 min/seed)
run_streaming(
    "source /content/miniconda3/etc/profile.d/conda.sh && conda activate layoutlmv3 && export PYTHONUNBUFFERED=1 && cd /content/layoutlmv3-1/layoutlmv3 && bash scripts/run_ablation_suite.sh cord base baseline",
    "Cell 5: CORD | BASE | BASELINE (3 seeds: 42, 123, 1993) — Est. ~18-22 min"
)

In [ ]:
# Cell 6: [BASE 3/10] FUNSD — Full Model (HPE + Token-Segment + Boundary Loss, 3 seeds)
# Estimated time on A100: ~7-9 minutes (3 seeds x ~2.5-3 min/seed)
run_streaming(
    "source /content/miniconda3/etc/profile.d/conda.sh && conda activate layoutlmv3 && export PYTHONUNBUFFERED=1 && cd /content/layoutlmv3-1/layoutlmv3 && bash scripts/run_ablation_suite.sh funsd base full",
    "Cell 6: FUNSD | BASE | FULL MODEL (3 seeds: 42, 123, 1993) — Est. ~7-9 min"
)

In [ ]:
# Cell 7: [BASE 4/10] CORD — Full Model (HPE + Token-Segment + Boundary Loss, 3 seeds)
# Estimated time on A100: ~20-25 minutes (3 seeds x ~7-8 min/seed)
run_streaming(
    "source /content/miniconda3/etc/profile.d/conda.sh && conda activate layoutlmv3 && export PYTHONUNBUFFERED=1 && cd /content/layoutlmv3-1/layoutlmv3 && bash scripts/run_ablation_suite.sh cord base full",
    "Cell 7: CORD | BASE | FULL MODEL (3 seeds: 42, 123, 1993) — Est. ~20-25 min"
)

In [ ]:
# Cell 8: [BASE 5/10] FUNSD — Ablation 1: Without Boundary Loss (HPE + Token-Segment only, 3 seeds)
# Estimated time on A100: ~7-9 minutes (3 seeds x ~2.5-3 min/seed)
run_streaming(
    "source /content/miniconda3/etc/profile.d/conda.sh && conda activate layoutlmv3 && export PYTHONUNBUFFERED=1 && cd /content/layoutlmv3-1/layoutlmv3 && bash scripts/run_ablation_suite.sh funsd base no_boundary",
    "Cell 8: FUNSD | BASE | WITHOUT BOUNDARY LOSS (3 seeds: 42, 123, 1993) — Est. ~7-9 min"
)

In [ ]:
# Cell 9: [BASE 6/10] CORD — Ablation 1: Without Boundary Loss (HPE + Token-Segment only, 3 seeds)
# Estimated time on A100: ~20-24 minutes (3 seeds x ~6.5-8 min/seed)
run_streaming(
    "source /content/miniconda3/etc/profile.d/conda.sh && conda activate layoutlmv3 && export PYTHONUNBUFFERED=1 && cd /content/layoutlmv3-1/layoutlmv3 && bash scripts/run_ablation_suite.sh cord base no_boundary",
    "Cell 9: CORD | BASE | WITHOUT BOUNDARY LOSS (3 seeds: 42, 123, 1993) — Est. ~20-24 min"
)

In [ ]:
# Cell 10: [BASE 7/10] FUNSD — Ablation 2: Without Token-Segment (HPE + Boundary Loss only, 3 seeds)
# Estimated time on A100: ~6-8 minutes (3 seeds x ~2-2.5 min/seed)
run_streaming(
    "source /content/miniconda3/etc/profile.d/conda.sh && conda activate layoutlmv3 && export PYTHONUNBUFFERED=1 && cd /content/layoutlmv3-1/layoutlmv3 && bash scripts/run_ablation_suite.sh funsd base no_segment",
    "Cell 10: FUNSD | BASE | WITHOUT TOKEN-SEGMENT (3 seeds: 42, 123, 1993) — Est. ~6-8 min"
)

In [ ]:
# Cell 11: [BASE 8/10] CORD — Ablation 2: Without Token-Segment (HPE + Boundary Loss only, 3 seeds)
# Estimated time on A100: ~18-22 minutes (3 seeds x ~6-7 min/seed)
run_streaming(
    "source /content/miniconda3/etc/profile.d/conda.sh && conda activate layoutlmv3 && export PYTHONUNBUFFERED=1 && cd /content/layoutlmv3-1/layoutlmv3 && bash scripts/run_ablation_suite.sh cord base no_segment",
    "Cell 11: CORD | BASE | WITHOUT TOKEN-SEGMENT (3 seeds: 42, 123, 1993) — Est. ~18-22 min"
)

In [ ]:
# Cell 12: [BASE 9/10] FUNSD — Ablation 3: Without HPE (Token-Segment + Boundary Loss only, 3 seeds)
# Estimated time on A100: ~7-9 minutes (3 seeds x ~2.5-3 min/seed)
run_streaming(
    "source /content/miniconda3/etc/profile.d/conda.sh && conda activate layoutlmv3 && export PYTHONUNBUFFERED=1 && cd /content/layoutlmv3-1/layoutlmv3 && bash scripts/run_ablation_suite.sh funsd base no_hpe",
    "Cell 12: FUNSD | BASE | WITHOUT HPE (3 seeds: 42, 123, 1993) — Est. ~7-9 min"
)

In [ ]:
# Cell 13: [BASE 10/10] CORD — Ablation 3: Without HPE (Token-Segment + Boundary Loss only, 3 seeds)
# Estimated time on A100: ~20-24 minutes (3 seeds x ~6.5-8 min/seed)
run_streaming(
    "source /content/miniconda3/etc/profile.d/conda.sh && conda activate layoutlmv3 && export PYTHONUNBUFFERED=1 && cd /content/layoutlmv3-1/layoutlmv3 && bash scripts/run_ablation_suite.sh cord base no_hpe",
    "Cell 13: CORD | BASE | WITHOUT HPE (3 seeds: 42, 123, 1993) — Est. ~20-24 min"
)

---
## Phase B: All `layoutlmv3-large` Runs (3 Seeds: `42, 123, 1993`)
Runs all `large` configurations after all `base` configurations have completed.

In [ ]:
# Cell 14: [LARGE 1/10] FUNSD — Baseline (Vanilla LayoutLMv3-large, 3 seeds)
# Estimated time on A100: ~12-15 minutes (3 seeds x ~4-5 min/seed)
run_streaming(
    "source /content/miniconda3/etc/profile.d/conda.sh && conda activate layoutlmv3 && export PYTHONUNBUFFERED=1 && cd /content/layoutlmv3-1/layoutlmv3 && bash scripts/run_ablation_suite.sh funsd large baseline",
    "Cell 14: FUNSD | LARGE | BASELINE (3 seeds: 42, 123, 1993) — Est. ~12-15 min"
)

In [ ]:
# Cell 15: [LARGE 2/10] CORD — Baseline (Vanilla LayoutLMv3-large, 3 seeds)
# Estimated time on A100: ~35-42 minutes (3 seeds x ~12-14 min/seed)
run_streaming(
    "source /content/miniconda3/etc/profile.d/conda.sh && conda activate layoutlmv3 && export PYTHONUNBUFFERED=1 && cd /content/layoutlmv3-1/layoutlmv3 && bash scripts/run_ablation_suite.sh cord large baseline",
    "Cell 15: CORD | LARGE | BASELINE (3 seeds: 42, 123, 1993) — Est. ~35-42 min"
)

In [ ]:
# Cell 16: [LARGE 3/10] FUNSD — Full Model (HPE + Token-Segment + Boundary Loss, 3 seeds)
# Estimated time on A100: ~13-16 minutes (3 seeds x ~4.5-5.5 min/seed)
run_streaming(
    "source /content/miniconda3/etc/profile.d/conda.sh && conda activate layoutlmv3 && export PYTHONUNBUFFERED=1 && cd /content/layoutlmv3-1/layoutlmv3 && bash scripts/run_ablation_suite.sh funsd large full",
    "Cell 16: FUNSD | LARGE | FULL MODEL (3 seeds: 42, 123, 1993) — Est. ~13-16 min"
)

In [ ]:
# Cell 17: [LARGE 4/10] CORD — Full Model (HPE + Token-Segment + Boundary Loss, 3 seeds)
# Estimated time on A100: ~38-45 minutes (3 seeds x ~13-15 min/seed)
run_streaming(
    "source /content/miniconda3/etc/profile.d/conda.sh && conda activate layoutlmv3 && export PYTHONUNBUFFERED=1 && cd /content/layoutlmv3-1/layoutlmv3 && bash scripts/run_ablation_suite.sh cord large full",
    "Cell 17: CORD | LARGE | FULL MODEL (3 seeds: 42, 123, 1993) — Est. ~38-45 min"
)

In [ ]:
# Cell 18: [LARGE 5/10] FUNSD — Ablation 1: Without Boundary Loss (HPE + Token-Segment only, 3 seeds)
# Estimated time on A100: ~13-16 minutes (3 seeds x ~4.5-5.5 min/seed)
run_streaming(
    "source /content/miniconda3/etc/profile.d/conda.sh && conda activate layoutlmv3 && export PYTHONUNBUFFERED=1 && cd /content/layoutlmv3-1/layoutlmv3 && bash scripts/run_ablation_suite.sh funsd large no_boundary",
    "Cell 18: FUNSD | LARGE | WITHOUT BOUNDARY LOSS (3 seeds: 42, 123, 1993) — Est. ~13-16 min"
)

In [ ]:
# Cell 19: [LARGE 6/10] CORD — Ablation 1: Without Boundary Loss (HPE + Token-Segment only, 3 seeds)
# Estimated time on A100: ~38-45 minutes (3 seeds x ~13-15 min/seed)
run_streaming(
    "source /content/miniconda3/etc/profile.d/conda.sh && conda activate layoutlmv3 && export PYTHONUNBUFFERED=1 && cd /content/layoutlmv3-1/layoutlmv3 && bash scripts/run_ablation_suite.sh cord large no_boundary",
    "Cell 19: CORD | LARGE | WITHOUT BOUNDARY LOSS (3 seeds: 42, 123, 1993) — Est. ~38-45 min"
)

In [ ]:
# Cell 20: [LARGE 7/10] FUNSD — Ablation 2: Without Token-Segment (HPE + Boundary Loss only, 3 seeds)
# Estimated time on A100: ~12-15 minutes (3 seeds x ~4-5 min/seed)
run_streaming(
    "source /content/miniconda3/etc/profile.d/conda.sh && conda activate layoutlmv3 && export PYTHONUNBUFFERED=1 && cd /content/layoutlmv3-1/layoutlmv3 && bash scripts/run_ablation_suite.sh funsd large no_segment",
    "Cell 20: FUNSD | LARGE | WITHOUT TOKEN-SEGMENT (3 seeds: 42, 123, 1993) — Est. ~12-15 min"
)

In [ ]:
# Cell 21: [LARGE 8/10] CORD — Ablation 2: Without Token-Segment (HPE + Boundary Loss only, 3 seeds)
# Estimated time on A100: ~35-42 minutes (3 seeds x ~12-14 min/seed)
run_streaming(
    "source /content/miniconda3/etc/profile.d/conda.sh && conda activate layoutlmv3 && export PYTHONUNBUFFERED=1 && cd /content/layoutlmv3-1/layoutlmv3 && bash scripts/run_ablation_suite.sh cord large no_segment",
    "Cell 21: CORD | LARGE | WITHOUT TOKEN-SEGMENT (3 seeds: 42, 123, 1993) — Est. ~35-42 min"
)

In [ ]:
# Cell 22: [LARGE 9/10] FUNSD — Ablation 3: Without HPE (Token-Segment + Boundary Loss only, 3 seeds)
# Estimated time on A100: ~13-16 minutes (3 seeds x ~4.5-5.5 min/seed)
run_streaming(
    "source /content/miniconda3/etc/profile.d/conda.sh && conda activate layoutlmv3 && export PYTHONUNBUFFERED=1 && cd /content/layoutlmv3-1/layoutlmv3 && bash scripts/run_ablation_suite.sh funsd large no_hpe",
    "Cell 22: FUNSD | LARGE | WITHOUT HPE (3 seeds: 42, 123, 1993) — Est. ~13-16 min"
)

In [ ]:
# Cell 23: [LARGE 10/10] CORD — Ablation 3: Without HPE (Token-Segment + Boundary Loss only, 3 seeds)
# Estimated time on A100: ~38-45 minutes (3 seeds x ~13-15 min/seed)
run_streaming(
    "source /content/miniconda3/etc/profile.d/conda.sh && conda activate layoutlmv3 && export PYTHONUNBUFFERED=1 && cd /content/layoutlmv3-1/layoutlmv3 && bash scripts/run_ablation_suite.sh cord large no_hpe",
    "Cell 23: CORD | LARGE | WITHOUT HPE (3 seeds: 42, 123, 1993) — Est. ~38-45 min"
)

In [ ]:
# Cell 24: Aggregate All 3-Seed Results into a Final Markdown & JSON Table
# Estimated time to finish this cell: ~2 seconds
import glob, json, os

log_dir = "/content/layoutlmv3-1/layoutlmv3/logs"
modes = ["baseline", "full", "no_boundary", "no_segment", "no_hpe"]
mode_labels = {
    "baseline": "Baseline (Vanilla LayoutLMv3)",
    "full": "Full Model (HPE + Token-Segment + Boundary Loss)",
    "no_boundary": "Ablation: w/o Boundary Loss",
    "no_segment": "Ablation: w/o Token-Segment",
    "no_hpe": "Ablation: w/o HPE",
}

print(f"| {'Variant':<7} | {'Dataset':<7} | {'Configuration':<48} | {'Precision (%)':<16} | {'Recall (%)':<16} | {'F1 (%)':<16} |")
print("|" + "-"*9 + "|" + "-"*9 + "|" + "-"*50 + "|" + "-"*18 + "|" + "-"*18 + "|" + "-"*18 + "|")

for variant in ["base", "large"]:
    for dataset in ["funsd", "cord"]:
        for mode in modes:
            summary_path = os.path.join(log_dir, f"{dataset}-{variant}-{mode}_3seed_summary.json")
            if not os.path.exists(summary_path):
                print(f"| {variant:<7} | {dataset.upper():<7} | {mode_labels[mode]:<48} | {'PENDING':<16} | {'PENDING':<16} | {'PENDING':<16} |")
                continue
            with open(summary_path, "r") as f:
                data = json.load(f)["metrics"]
            p_str = f"{data['eval_precision']['mean']*100:.2f} ± {data['eval_precision']['std']*100:.2f}"
            r_str = f"{data['eval_recall']['mean']*100:.2f} ± {data['eval_recall']['std']*100:.2f}"
            f1_str = f"{data['eval_f1']['mean']*100:.2f} ± {data['eval_f1']['std']*100:.2f}"
            print(f"| {variant:<7} | {dataset.upper():<7} | {mode_labels[mode]:<48} | {p_str:<16} | {r_str:<16} | {f1_str:<16} |")